# Del texto de Borges al próximo token

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---
| | |
|---|---|
| **Problema** | Un modelo generativo no recibe un texto: recibe ejemplos numéricos de qué token debería seguir. |
| **Producto** | Una cadena verificable desde un fragmento de Borges hasta pares de entrada y objetivo para un modelo causal. |
| **Recursos** | Computadora local, 1 GB de memoria y las dependencias de requirements.txt. No descarga modelos. |

El laboratorio GPT-2 que sigue puede ajustar un modelo a un corpus. Antes necesitamos abrir la transformación que hace posible ese ajuste: convertir un texto en muchos ejemplos de predicción del próximo token.

**¿Cómo se transforma Tlön, Uqbar, Orbis Tertius en pares que un Transformer causal puede usar para aprender?**

> **Cómo se trabaja.** Explorás el recorrido con el texto completo y con una ventana pequeña que se puede leer. Al final de cada bloque hay un microlaboratorio con una comprobación.


In [ ]:
from pathlib import Path

import numpy as np
import tiktoken

import visualizacion_texto as visual

RUTA_TEXTO = Path("tlon-uqbar-orbis-tertius.txt")
tokenizador = tiktoken.get_encoding("gpt2")

print(f"tiktoken {tiktoken.__version__}")
print(f"Tokenizador: {tokenizador.name}")


## 1. Un texto real, con una procedencia que importa

Trabajamos con una copia local de Tlön, Uqbar, Orbis Tertius, de Jorge Luis Borges. El archivo fue incorporado antes a la unidad como texto de estudio. Esta lectura no evalúa a Borges ni afirma que un modelo aprenda su estilo: muestra qué datos recibe un modelo generativo.


In [ ]:
texto_completo = RUTA_TEXTO.read_text(encoding="utf-8")
fragmento = texto_completo[:400]

print(f"Caracteres en el archivo: {len(texto_completo):,}")
print()
print(fragmento)


> **Para mirar.** La computadora todavía ve caracteres. Ni las palabras ni sus límites son una unidad natural para el modelo: el siguiente paso decide cómo partir esta secuencia.

## 2. BPE evita un vocabulario cerrado

GPT-2 usa codificación por pares de bytes, o BPE. Un token puede coincidir con una palabra, con parte de una palabra, con puntuación o con espacios codificados. Por eso es importante mirar los tokens concretos y no asumir que una palabra siempre equivale a un token.


In [ ]:
texto_corto = "Tlön, Uqbar, Orbis Tertius"
ids_cortos = tokenizador.encode(texto_corto)
tokens_cortos = []

for identificador in ids_cortos:
    tokens_cortos.append(tokenizador.decode([identificador]))

notas = []
for identificador in ids_cortos:
    notas.append(str(identificador))

visual.dibujar_fichas(
    piezas=tokens_cortos,
    titulo="BPE separa el titulo en tokens",
    notas=notas,
)

texto_reconstruido = tokenizador.decode(ids_cortos)
print(list(zip(tokens_cortos, ids_cortos)))
visual.comprobar(
    texto_reconstruido == texto_corto,
    "revisa que uses el mismo tokenizador para codificar y decodificar.",
)


> **Para mirar.** Los identificadores no ordenan palabras por significado: son direcciones en el vocabulario del tokenizador. El texto puede reconstruirse porque el mismo tokenizador sabe deshacer esa codificación.

## 3. Una ventana convierte la lectura en una tarea de predicción

El modelo no recibe el libro entero como una sola entrada. Tomamos una ventana de tokens y la desplazamos un lugar. La entrada contiene lo que ya se conoce; el objetivo contiene exactamente el token que debería seguir en cada posición.


In [ ]:
fragmento = texto_completo[:400]
ids_fragmento = tokenizador.encode(fragmento)
tokens_fragmento = []

for identificador in ids_fragmento:
    tokens_fragmento.append(tokenizador.decode([identificador]))

largo_ventana = 6
entrada_tokens = tokens_fragmento[:largo_ventana]
objetivo_tokens = tokens_fragmento[1:largo_ventana + 1]

colores_entrada = []
notas_posicion = []
for posicion in range(largo_ventana):
    colores_entrada.append(visual.AZUL)
    notas_posicion.append(f"posicion {posicion}")

visual.dibujar_fichas(
    piezas=entrada_tokens,
    titulo="Entrada: lo que el modelo ya puede usar",
    colores=colores_entrada,
    notas=notas_posicion,
)
colores_objetivo = []
for posicion in range(largo_ventana):
    colores_objetivo.append(visual.NARANJA)

visual.dibujar_fichas(
    piezas=objetivo_tokens,
    titulo="Objetivo: el token siguiente en cada posicion",
    colores=colores_objetivo,
    notas=notas_posicion,
)


> **Para mirar.** No hay seis predicciones independientes: el objetivo de la primera posición es el segundo token de la entrada, y así sucesivamente. Este desplazamiento de un lugar es la tarea causal que GPT-2 aprende.

### Microlaboratorio 1 — Desplazar la ventana

**Tarea.** Construí el objetivo que corresponde a una entrada de cuatro tokens.  
**Producto.** Una lista que empieza en el segundo token de entrada y suma el quinto token del fragmento.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) | los cuatro tokens de entrada | el corte inicial de tokens_fragmento |
| (2) | los cuatro objetivos desplazados | el corte que comienza una posición después |

> **Ayuda 1.** Compará la posición inicial de entrada_tokens y objetivo_tokens.  
> **Ayuda 2.** Un corte de cuatro elementos que comienza en 1 termina antes de 5.


In [ ]:
entrada_practica = []  # COMPLETAR (1)
objetivo_practica = []  # COMPLETAR (2)

visual.comprobar(
    entrada_practica == tokens_fragmento[:4]
    and objetivo_practica == tokens_fragmento[1:5],
    "revisa ambos cortes: deben tener cuatro tokens y estar desplazados un lugar.",
)


## 4. La posición no puede perderse

Attention relaciona todos los tokens disponibles, pero por sí sola no sabe cuál apareció primero. Por eso cada posición aporta un vector propio que se suma al embedding del token. Esta matriz es una demostración sintética: cada fila representa una posición y cada columna una dimensión pequeña; no son los embeddings reales de GPT-2.


In [ ]:
posiciones = ["0", "1", "2", "3", "4", "5"]
posiciones_sinteticas = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
    [1, 1, 0, 0],
    [0, 1, 1, 0],
])

visual.dibujar_mapa(
    matriz=posiciones_sinteticas,
    filas=posiciones,
    columnas=["d1", "d2", "d3", "d4"],
    titulo="Un ejemplo pequeno de informacion posicional",
)


> **Para mirar.** La matriz no codifica significado de palabras. Solo ilustra que las mismas dimensiones pueden recibir una señal distinta según el lugar que ocupa el token. En 011 vimos que la atención causal bloquea el futuro; aquí agregamos qué lugar tiene cada token que queda disponible.

## 5. Del ejemplo al ciclo generativo

Durante el ajuste, el modelo compara sus predicciones con los objetivos desplazados y modifica sus pesos. Durante la generación, no tiene objetivos: toma un prefijo, calcula probabilidades para el próximo token, elige uno según una regla de muestreo y vuelve a empezar.


In [ ]:
visual.dibujar_ciclo()

candidatos = [" el", " una", " que", " de"]
probabilidades = [0.42, 0.26, 0.19, 0.13]
visual.dibujar_barras(
    nombres=candidatos,
    valores=probabilidades,
    titulo="Ejemplo sintetico de candidatos del proximo token",
    color=visual.NARANJA,
)


> **Para mirar.** Estas probabilidades son inventadas para mostrar la forma de una distribución. En el laboratorio GPT-2 van a inspeccionar candidatos calculados por un modelo real y a distinguir entre cambiar el muestreo y cambiar los pesos mediante ajuste.

## Qué te llevás

- BPE convierte el texto en tokens e identificadores sin depender de un vocabulario de palabras completo.
- Una ventana y su objetivo desplazado construyen la tarea de predecir el próximo token.
- Los embeddings de token y de posición pasan a un Transformer causal como el que abrimos en 011.

**Sigue:** en el laboratorio GPT-2, el modelo base recibirá bloques de este tipo; van a decidir qué corpus usar, ajustar el modelo y evaluar qué evidencia queda del cambio.
